In [ ]:
# Load required libraries
library(ggplot2)
library(dplyr)
library(tidyr)
library(stringr)
library(data.table)
library(jsonlite)

# Set timeout option
options(timeout = 3600)

# Load input parameters from JSON
json_input <- fromJSON("galaxy_inputs/galaxy_inputs.json")

# Extract parameters from JSON
data_path      <- json_input$data$path
selected_cols  <- json_input$colXchose
plot_mode      <- json_input$plotXmode
filter_keyword <- json_input$keyword
filter_keyword <- gsub(" ", ".", filter_keyword)
plot_title     <- json_input$title
x_axis_label   <- json_input$x
y_axis_label   <- json_input$y
apply_log      <- json_input$log
reference_year <- json_input$year
# Load dataset
raw_data <- fread(data_path)

# Clean column names to ensure uniqueness and valid syntax
colnames(raw_data) <- make.names(names(raw_data), unique = TRUE)

# Convert numeric columns (except Year) by replacing commas with dots
clean_data <- raw_data %>%
  mutate(across(
    .cols = -Year,
    .fns = ~ as.double(str_replace_all(., ",", "."))
  ))

# Select relevant columns based on keyword or manual selection
if (!is.null(filter_keyword) && filter_keyword != "") {
  selected_data <- clean_data %>%
    select(Year, matches(filter_keyword))
} else {
  selected_indices <- as.numeric(unlist(strsplit(selected_cols, ",")))
  selected_data <- clean_data %>%
    select(Year, selected_indices)
}

# Apply log transformation if requested
if (apply_log) {
  logged_data <- selected_data %>%
    mutate(across(
      .cols = matches("pairs"),
      .fns = ~ {
        ref_value <- selected_data %>%
          filter(Year == reference_year) %>%
          pull(cur_column())
        
        if (is.na(ref_value)) return(NA_real_)
        
        normalized <- . / ref_value
        log_value <- log(normalized)
        return(log_value)
      },
      .names = "log_{.col}"
    )) %>%
    select(Year, starts_with("log_"))
  
  selected_data <- logged_data
}

# Reshape data to long format for plotting
long_data <- selected_data %>%
  pivot_longer(cols = -Year, names_to = "Species", values_to = "Value")

# Generate plots
if (plot_mode == "merged_view") {
  png("outputs/collection/indicator_plot.png")
  print(
    ggplot(long_data, aes(x = Year, y = Value, color = Species)) +
      geom_line(na.rm = TRUE) +
      labs(
        title = plot_title,
        x = x_axis_label,
        y = y_axis_label
      ) +
      theme_minimal()
  )
  dev.off()
} else {
  for (species in unique(long_data$Species)) {
    species_data <- filter(long_data, Species == species)
    output_filename <- paste0("outputs/collection/plot_", str_replace_all(species, "[^a-zA-Z0-9]", "_"), ".png")
    
    png(output_filename, width = 800, height = 600)
    print(
      ggplot(species_data, aes(x = Year, y = Value)) +
        geom_line(color = "steelblue", na.rm = TRUE) +
        labs(
          title = paste("Trend -", species),
          x = "Year",
          y = "Log(Indicator)"
        ) +
        theme_minimal()
    )
    dev.off()
  }
}
